# RiceShift
## NASA-Powered Rotation Advisor for California Rice Farms

**NASA Space Apps Challenge 2026 — Field Shift**  
**Team Hachi**

RiceShift is a decision-support prototype for California rice agriculture. It combines historical crop management, NASA Earth observations, soil properties, and agricultural outcomes in one transparent workflow.

### Data used
- **NASA MODIS** — vegetation, evapotranspiration, land-surface temperature, spectral water indicators
- **NASA SMAP** — surface soil moisture
- **USDA Cropland Data Layer (CDL)** — crop and rotation history
- **USDA NASS** — rice yield, harvested area, production
- **USDA NRCS SSURGO** — top-30 cm soil properties

### Experimental questions
1. **What has happened?** — How did rice and crop rotation change from 2007–2025?
2. **Under what environmental conditions?** — How do winter and rice-growing-season conditions differ?
3. **How are those signals associated with agricultural outcomes?** — How do environmental indicators relate to observed rice yield?

### Candidate strategies
- **Continuous Rice**
- **Rice → Winter Vetch → Rice**
- **Rice → Winter Cover Crop Mix → Rice**

> This notebook is an exploratory decision-support prototype. It does not claim causal agronomic effects and is not a replacement for field validation.


# 1. Load the analysis bundle

The raw archive remains on AWS EC2. This Colab notebook uses only the compact, analysis-ready outputs needed for exploration, visualization, and demonstration.


In [9]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil
import pandas as pd
import numpy as np

ZIP = Path("/content/drive/MyDrive/プレゼン発表/NASA_RiceShift/RiceShift_colab_bundle.zip")
WORK = Path("/content/RiceShift")

if WORK.exists():
    shutil.rmtree(WORK)

WORK.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(str(ZIP), str(WORK))

ROOT = WORK / "RiceShift_colab_bundle"

print("ZIP :", ZIP.exists())
print("ROOT:", ROOT.exists())
print(list(ROOT.iterdir()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ZIP : True
ROOT: True
[PosixPath('/content/RiceShift/RiceShift_colab_bundle/scripts'), PosixPath('/content/RiceShift/RiceShift_colab_bundle/README.md'), PosixPath('/content/RiceShift/RiceShift_colab_bundle/data')]


## 1.1 Load analysis-ready datasets

The bundle contains monthly, seasonal, annual, crop-history, SMAP, NASS, and SSURGO tables prepared during the data-engineering phase.


In [10]:
monthly = pd.read_parquet(
    ROOT / "data/step24/monthly_features.parquet"
)

seasonal = pd.read_parquet(
    ROOT / "data/step24/seasonal_features.parquet"
)

winter = pd.read_parquet(
    ROOT / "data/step24/winter_cover_features.parquet"
)

full_year = pd.read_parquet(
    ROOT / "data/step24/full_year_features.parquet"
)

nass = pd.read_parquet(
    ROOT / "data/step25/model_yearly.parquet"
)

nass_monthly = pd.read_parquet(
    ROOT / "data/step25/monthly_features_with_nass.parquet"
)

smap_daily = pd.read_parquet(
    ROOT / "data/step27ex/smap_daily.parquet"
)

smap_monthly = pd.read_parquet(
    ROOT / "data/step27ex/smap_monthly.parquet"
)

smap_seasonal = pd.read_parquet(
    ROOT / "data/step27ex/smap_seasonal.parquet"
)

rotation = pd.read_csv(
    ROOT / "data/step28/yearly_group_counts.csv"
)

rotation_summary = pd.read_csv(
    ROOT / "data/step28/rotation_class_summary.csv"
)

soil = pd.read_parquet(
    ROOT / "data/step29/ssurgo_mapunit_top30cm.parquet"
)

print("monthly         :", monthly.shape)
print("seasonal        :", seasonal.shape)
print("winter          :", winter.shape)
print("full_year       :", full_year.shape)
print("NASS            :", nass.shape)
print("NASS monthly    :", nass_monthly.shape)
print("SMAP daily      :", smap_daily.shape)
print("SMAP monthly    :", smap_monthly.shape)
print("SMAP seasonal   :", smap_seasonal.shape)
print("rotation yearly :", rotation.shape)
print("rotation summary:", rotation_summary.shape)
print("soil            :", soil.shape)

monthly         : (228, 23)
seasonal        : (19, 22)
winter          : (18, 22)
full_year       : (19, 22)
NASS            : (19, 26)
NASS monthly    : (228, 27)
SMAP daily      : (3820, 15)
SMAP monthly    : (129, 9)
SMAP seasonal   : (21, 9)
rotation yearly : (19, 12)
rotation summary: (4, 3)
soil            : (4977, 19)


## 1.2 Data integrity check

Before interpreting any result, verify row counts, duplicates, missing values, time coverage, and available variables.

Expected core coverage:
- MODIS / CDL / NASS: **2007–2025**
- Winter MODIS features: **2008–2025**
- SMAP: **2015–2025**
- CDL crop history: **19 years**
- SSURGO top-30 cm soil profiles: **4,977 map units**


In [11]:
datasets = {
    "monthly": monthly,
    "seasonal": seasonal,
    "winter": winter,
    "full_year": full_year,
    "nass": nass,
    "smap_daily": smap_daily,
    "smap_monthly": smap_monthly,
    "smap_seasonal": smap_seasonal,
    "rotation": rotation,
    "soil": soil,
}

for name, df in datasets.items():
    print("\n" + "="*70)
    print(name, df.shape)
    print("="*70)

    print("Duplicates:", df.duplicated().sum())
    print("Missing cells:", int(df.isna().sum().sum()))

    if "year" in df.columns:
        print("Year:", df["year"].min(), "-", df["year"].max())

    if "month" in df.columns:
        print("Months:", sorted(df["month"].dropna().unique()))

    print("\nColumns:")
    print(list(df.columns))


monthly (228, 23)
Duplicates: 0
Missing cells: 0
Year: 2007 - 2025
Months: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

Columns:
['year', 'month', 'blue', 'et', 'evi', 'fpar', 'green', 'lai', 'lst_day_c', 'lst_night_c', 'mndwi', 'ndmi', 'ndvi', 'ndwi', 'nir', 'pet', 'red', 'swir1', 'swir2', 'swir_1230', 'cdl_rice_pixels', 'cdl_rice_area_hectares', 'cdl_rice_area_acres']

seasonal (19, 22)
Duplicates: 0
Missing cells: 0
Year: 2007 - 2025

Columns:
['year', 'blue', 'et', 'evi', 'fpar', 'green', 'lai', 'lst_day_c', 'lst_night_c', 'mndwi', 'ndmi', 'ndvi', 'ndwi', 'nir', 'pet', 'red', 'swir1', 'swir2', 'swir_1230', 'cdl_rice_pixels', 'cdl_rice_area_hectares', 'cdl_rice_area_acres']

winter (18, 22)
Duplicates: 0
Missing cells: 0
Year: 2008 - 2025

Columns:
['year', 'blue', 'et', 'evi', 'fpar', 'green', 'lai', 'lst_day_c', 'lst_night_c', 'mndwi', 'ndmi', 'ndvi', 'ndwi', 'nir',

In [12]:
# ============================================================
# EDA 2 — Key columns / missingness / candidate targets
# ============================================================

import pandas as pd
import numpy as np

def compact_summary(name, df):
    print("\n" + "="*80)
    print(name, df.shape)
    print("="*80)

    miss = (
        df.isna().mean()
          .mul(100)
          .sort_values(ascending=False)
    )

    print("\nColumns with missing values (%)")
    print(miss[miss > 0].head(20).round(1))

    print("\nNumeric columns:")
    print(list(df.select_dtypes(include=np.number).columns))


compact_summary("STEP24 MONTHLY", monthly)
compact_summary("STEP24 WINTER", winter)
compact_summary("STEP25 NASS MODEL", nass)
compact_summary("STEP27 SMAP SEASONAL", smap_seasonal)
compact_summary("STEP28 ROTATION", rotation)
compact_summary("STEP29 SOIL", soil)


print("\n" + "="*80)
print("NASS TARGET CANDIDATES")
print("="*80)

target_words = [
    "yield",
    "production",
    "harvest",
    "area",
    "acre",
    "price"
]

target_candidates = [
    c for c in nass.columns
    if any(w in c.lower() for w in target_words)
]

print(target_candidates)

if target_candidates:
    display(nass[["year"] + target_candidates])



STEP24 MONTHLY (228, 23)

Columns with missing values (%)
Series([], dtype: float64)

Numeric columns:
['year', 'month', 'blue', 'et', 'evi', 'fpar', 'green', 'lai', 'lst_day_c', 'lst_night_c', 'mndwi', 'ndmi', 'ndvi', 'ndwi', 'nir', 'pet', 'red', 'swir1', 'swir2', 'swir_1230', 'cdl_rice_pixels', 'cdl_rice_area_hectares', 'cdl_rice_area_acres']

STEP24 WINTER (18, 22)

Columns with missing values (%)
Series([], dtype: float64)

Numeric columns:
['year', 'blue', 'et', 'evi', 'fpar', 'green', 'lai', 'lst_day_c', 'lst_night_c', 'mndwi', 'ndmi', 'ndvi', 'ndwi', 'nir', 'pet', 'red', 'swir1', 'swir2', 'swir_1230', 'cdl_rice_pixels', 'cdl_rice_area_hectares', 'cdl_rice_area_acres']

STEP25 NASS MODEL (19, 26)

Columns with missing values (%)
Series([], dtype: float64)

Numeric columns:
['year', 'blue', 'et', 'evi', 'fpar', 'green', 'lai', 'lst_day_c', 'lst_night_c', 'mndwi', 'ndmi', 'ndvi', 'ndwi', 'nir', 'pet', 'red', 'swir1', 'swir2', 'swir_1230', 'cdl_rice_pixels', 'cdl_rice_area_hectares

,year,cdl_rice_area_hectares,cdl_rice_area_acres,nass_harvested_acres,nass_planted_acres,nass_production_cwt,nass_yield_lb_acre
0,2007,249413.6064,616314.443526,533000.0,534000.0,43684000.0,8200.0
1,2008,218668.5000,540341.631074,517000.0,519000.0,43030000.0,8320.0
2,2009,233898.2100,577975.064065,556000.0,561000.0,47804000.0,8600.0
3,2010,239326.2900,591388.141856,553000.0,558000.0,44326000.0,8020.0
4,2011,239537.6100,591910.324948,580000.0,585000.0,48402000.0,8350.0
5,2012,232622.5500,574822.839556,557000.0,562000.0,45413000.0,8150.0
6,2013,229749.4800,567723.328973,562000.0,567000.0,47641000.0,8480.0
7,2014,189192.0600,467503.761569,442000.0,445000.0,37936000.0,8580.0
8,2015,177279.3000,438066.690527,426000.0,429000.0,37877000.0,8890.0
9,2016,222758.8200,550449.031913,536000.0,541000.0,47394000.0,8840.0


### Data integrity interpretation

The primary annual and monthly analysis tables are duplicate-free and complete for their configured periods.

SMAP daily observations include expected gaps caused by satellite spatial/temporal coverage. RiceShift therefore uses the **monthly and seasonal SMAP aggregates** for the main experiment.

The principal agricultural outcome used below is **`nass_yield_lb_acre`**, the annual NASS rice yield in pounds per acre.


# 2. Experiment 1 — 19-Year Crop History

### Question
**How has the Sacramento Valley crop landscape changed from 2007–2025?**

USDA CDL provides the management-history layer of RiceShift. The goal is to distinguish persistent rice production from years containing fallow land, winter annuals, summer annuals, forage crops, and other rotations.


In [ ]:
# ============================================================
# EXPERIMENT 1 — 19-Year Crop History
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path

FIG_DIR = Path('/content/RiceShift/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

crop_cols = [
    'rice',
    'fallow',
    'summer_annual',
    'winter_annual',
    'alfalfa',
    'grasses',
]

rotation_plot = rotation.set_index('year')[crop_cols]

display(rotation)

ax = rotation_plot.plot(figsize=(12, 6), linewidth=2)
ax.set_title('Sacramento Valley Crop History (2007–2025)')
ax.set_xlabel('Year')
ax.set_ylabel('CDL Pixel Count')
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(FIG_DIR / '01_crop_history_2007_2025.png', dpi=180, bbox_inches='tight')
plt.show()


### Interpretation

USDA CDL provides a 19-year record of crop-history patterns. RiceShift uses this history to identify whether the agricultural landscape remained dominated by rice or experienced alternative crops, fallow periods, or rotational behavior.

This gives RiceShift the **historical management context** needed before evaluating environmental conditions.


# 3. Experiment 2 — NASA Sees the Agricultural Seasons

### Question
**Were winter field conditions different from rice-growing-season conditions?**

RiceShift treats the agricultural year as two distinct windows:
- **Rice season:** April–November
- **Winter cover-crop window:** November of the previous year through March of the target year

NASA SMAP provides soil-moisture information that crop-history data alone cannot provide.


In [ ]:
# ============================================================
# EXPERIMENT 2 — NASA SMAP Seasonal Soil Moisture
# ============================================================

rice_smap = (
    smap_seasonal[smap_seasonal['season'] == 'rice_apr_nov'][['year', 'soil_moisture_mean']]
    .rename(columns={'soil_moisture_mean': 'Rice season'})
)

winter_smap = (
    smap_seasonal[smap_seasonal['season'] == 'winter_cover_nov_mar'][['year', 'soil_moisture_mean']]
    .rename(columns={'soil_moisture_mean': 'Winter cover season'})
)

smap_compare = (
    rice_smap
    .merge(winter_smap, on='year', how='outer')
    .sort_values('year')
)

display(smap_compare)

ax = smap_compare.set_index('year').plot(figsize=(11, 5), marker='o', linewidth=2)
ax.set_title('NASA SMAP Soil Moisture by Agricultural Season')
ax.set_xlabel('Year')
ax.set_ylabel('Mean Soil Moisture (m³/m³)')
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(FIG_DIR / '02_smap_seasonal_soil_moisture.png', dpi=180, bbox_inches='tight')
plt.show()


### Why this matters

Crop-history data tells us **what farmers did**. NASA SMAP adds information about **the environmental conditions in which those decisions occurred**.

RiceShift separately evaluates soil moisture during the rice-growing season and the preceding winter cover-crop window. A winter rotation must be feasible under winter field conditions, not only under summer rice conditions.


# 4. Experiment 3 — Integrating NASA Conditions with Agricultural Outcomes

### Question
**How do NASA-observed seasonal conditions align with observed rice yield?**

For this experiment, RiceShift combines:
- rice-season MODIS indicators
- winter-season MODIS indicators
- NASA SMAP rice-season soil moisture
- NASA SMAP winter soil moisture
- USDA NASS rice yield

Because a complete winter SMAP window begins in 2016, the common NASA comparison period is expected to be **2016–2025**.


In [ ]:
# ============================================================
# EXPERIMENT 3 — Integrated Annual Analysis
# ============================================================

rice_env = nass[
    [
        'year', 'ndvi', 'evi', 'lai', 'et', 'pet', 'lst_day_c',
        'nass_yield_lb_acre', 'nass_harvested_acres'
    ]
].copy()

rice_env = rice_env.rename(columns={
    'ndvi': 'rice_ndvi',
    'evi': 'rice_evi',
    'lai': 'rice_lai',
    'et': 'rice_et',
    'pet': 'rice_pet',
    'lst_day_c': 'rice_lst_day_c',
})

winter_env = winter[['year', 'ndvi', 'evi', 'et', 'lst_day_c']].copy()
winter_env = winter_env.rename(columns={
    'ndvi': 'winter_ndvi',
    'evi': 'winter_evi',
    'et': 'winter_et',
    'lst_day_c': 'winter_lst_day_c',
})

smap_rice = rice_smap.rename(columns={'Rice season': 'smap_rice'})
smap_win = winter_smap.rename(columns={'Winter cover season': 'smap_winter'})

integrated = (
    rice_env
    .merge(winter_env, on='year', how='left')
    .merge(smap_rice, on='year', how='left')
    .merge(smap_win, on='year', how='left')
)

submission_period = integrated.dropna(subset=['smap_rice', 'smap_winter']).copy()

display(submission_period)

print('Common NASA analysis period:',
      submission_period['year'].min(), '-', submission_period['year'].max())
print('Rows:', len(submission_period))


# 5. Exploratory Relationships with Rice Yield

The common NASA period contains only a small number of complete winter-to-rice seasons. RiceShift therefore uses **Spearman rank correlation** as an exploratory diagnostic rather than fitting a complex predictive model.

The purpose is to identify which environmental indicators may be useful as context for a decision-support system.


In [ ]:
# ============================================================
# EXPERIMENT 4 — Exploratory Associations with NASS Yield
# ============================================================

features = [
    'rice_ndvi',
    'rice_evi',
    'rice_lai',
    'rice_et',
    'rice_lst_day_c',
    'winter_ndvi',
    'winter_evi',
    'winter_et',
    'winter_lst_day_c',
    'smap_rice',
    'smap_winter',
]

corr = (
    submission_period[['nass_yield_lb_acre'] + features]
    .corr(method='spearman')['nass_yield_lb_acre']
    .drop('nass_yield_lb_acre')
    .sort_values(key=abs, ascending=False)
)

corr_table = corr.rename('Spearman correlation with NASS rice yield').to_frame()
display(corr_table)

top_corr = corr.head(8).sort_values()

plt.figure(figsize=(9, 5))
top_corr.plot(kind='barh')
plt.axvline(0, linewidth=1)
plt.title('Environmental Indicators Associated with California Rice Yield\nExploratory Spearman Correlations')
plt.xlabel('Spearman correlation with NASS yield')
plt.ylabel('')
plt.tight_layout()
plt.savefig(FIG_DIR / '03_exploratory_yield_correlations.png', dpi=180, bbox_inches='tight')
plt.show()


### Important interpretation

This is an **exploratory association analysis, not a causal experiment**.

The common SMAP period contains only ten complete winter-to-rice seasons (2016–2025). RiceShift therefore does **not** claim that any individual NASA variable causes higher or lower yield.

Instead, these relationships are used as environmental context for the decision-support prototype.


# 6. Soil Context — SSURGO Top-30 cm Profile

RiceShift also keeps a soil-suitability layer. SSURGO properties are summarized over the top 30 cm using the dominant component of each map unit.

This experiment does not force missing soil attributes to zero; it summarizes the available physical and chemical properties transparently.


In [ ]:
# ============================================================
# SOIL CONTEXT — SSURGO TOP-30 CM
# ============================================================

soil_features = [
    'ph1to1h2o_r_0_30cm',
    'ec_r_0_30cm',
    'sar_r_0_30cm',
    'ksat_r_0_30cm',
    'sandtotal_r_0_30cm',
    'silttotal_r_0_30cm',
    'claytotal_r_0_30cm',
    'om_r_0_30cm',
]

soil_summary = soil[soil_features].describe().T
soil_summary['missing_%'] = soil[soil_features].isna().mean().mul(100).round(1)

display(soil_summary)


# 7. RiceShift Recommendation Prototype

RiceShift is designed to compare management options rather than prescribe one universal rotation.

The prototype considers four evidence layers:
1. **Historical crop rotation** — USDA CDL
2. **Winter and rice-season environmental conditions** — NASA MODIS + NASA SMAP
3. **Soil suitability** — USDA NRCS SSURGO
4. **Observed agricultural outcomes** — USDA NASS

The current hackathon prototype compares three candidate strategies.


In [ ]:
# ============================================================
# RICESHIFT STRATEGY COMPARISON
# ============================================================

strategies = pd.DataFrame({
    'Strategy': [
        'Continuous Rice',
        'Rice → Winter Vetch → Rice',
        'Rice → Winter Cover Crop Mix → Rice',
    ],
    'Primary consideration': [
        'Maintain rice where historical production and seasonal conditions remain suitable',
        'Evaluate winter rotation opportunity with nitrogen-fixing cover potential',
        'Evaluate diversified winter cover where soil protection and rotation are desirable',
    ],
    'Evidence used': [
        'CDL history + rice-season MODIS + SMAP + NASS',
        'CDL rotation history + winter MODIS + winter SMAP + SSURGO',
        'CDL rotation history + winter MODIS + winter SMAP + SSURGO',
    ],
    'Prototype status': [
        'Decision-support candidate',
        'Decision-support candidate',
        'Decision-support candidate',
    ],
})

display(strategies)


## Prototype decision logic

At this stage, RiceShift is intentionally conservative:

- It **does not** claim that vetch or a cover-crop mix is universally superior.
- It uses NASA observations to characterize winter and rice-season conditions.
- It uses crop history and soil properties to provide local context.
- Final agronomic recommendations would require field-level validation, management constraints, and economic information.

This makes RiceShift a **transparent, data-driven screening framework** rather than a black-box prescription engine.


# 8. Experimental Conclusion

RiceShift demonstrates how public Earth-observation and agricultural datasets can be combined into one decision-support workflow.

### What each dataset contributes

**USDA CDL**  
→ What was planted and how the landscape was historically rotated.

**NASA MODIS**  
→ Vegetation, evapotranspiration, surface temperature, and water-related spectral conditions.

**NASA SMAP**  
→ Whether winter and rice-growing seasons were relatively wet or dry.

**USDA NRCS SSURGO**  
→ Soil physical and chemical suitability.

**USDA NASS**  
→ Observed agricultural outcomes such as yield and harvested area.

### RiceShift concept

Rather than recommending the same rotation everywhere, RiceShift evaluates **management history + soil conditions + seasonal Earth observations + agricultural outcomes** before comparing candidate rotation strategies.

### Current limitation

This hackathon prototype uses regional and historical observational data. It is intended as a foundation for future field-level validation, economic analysis, and operational farm recommendations.


# 9. Export Demo Figures

The three primary figures generated by this notebook are stored in `/content/RiceShift/figures/` and can be copied directly to the Google Drive project folder for the presentation, GitHub README, or short screen-recorded demonstration.


In [ ]:
# ============================================================
# EXPORT FIGURES TO GOOGLE DRIVE
# ============================================================

DRIVE_FIG_DIR = Path('/content/drive/MyDrive/プレゼン発表/NASA_RiceShift/figures')
DRIVE_FIG_DIR.mkdir(parents=True, exist_ok=True)

for fig in sorted(FIG_DIR.glob('*.png')):
    shutil.copy2(fig, DRIVE_FIG_DIR / fig.name)
    print('Copied:', fig.name)

print('Figure folder:', DRIVE_FIG_DIR)


# 10. Super Intelligence (SI) Use Disclosure

ChatGPT was used as an **SI-assisted development tool** for code drafting, debugging, data-pipeline review, notebook organization, and documentation.

The underlying scientific and agricultural datasets were obtained from the cited NASA and USDA sources. SI assistance did not replace the original source data. The final interpretations and project design were reviewed and selected by the project author.

No NASA logos, mission identifiers, or other NASA branding were generated or modified by SI tools in this notebook.
